# 4. Histopathology image analysis (H&E)

Color deconvolution and nuclei segmentation of a melanoma H&E image (CC BY 4.0, Weiss et al., BMC Cancer 2015). Melanin can confound H&E deconvolution; pixel measures are not calibrated to microns.

In [ ]:
import requests, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
from scipy import ndimage as ndi
from skimage import color, filters, morphology, measure, exposure, segmentation, feature
URL = "https://upload.wikimedia.org/wikipedia/commons/1/12/Histopathology_of_nodular_melanoma.jpg"
raw = requests.get(URL, timeout=60, headers={"User-Agent": "anveshar-research/1.0 (https://digvijayky.com)"}).content
img = np.array(Image.open(BytesIO(raw)).convert("RGB")); print("image:", img.shape)

## Color deconvolution (hematoxylin = nuclei, eosin = cytoplasm/stroma)

In [ ]:
hed = color.rgb2hed(img)
h = exposure.rescale_intensity(hed[:, :, 0], out_range=(0, 1)); e = exposure.rescale_intensity(hed[:, :, 1], out_range=(0, 1))
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(img); ax[0].set_title("H&E original")
ax[1].imshow(h, cmap="magma"); ax[1].set_title("Hematoxylin")
ax[2].imshow(e, cmap="viridis"); ax[2].set_title("Eosin")
[a.axis("off") for a in ax]; plt.tight_layout()

## Nuclei segmentation (watershed)

In [ ]:
hs = filters.gaussian(h, 1.0); mask = hs > filters.threshold_otsu(hs)
mask = ndi.binary_fill_holes(morphology.remove_small_objects(mask, 8))
dist = ndi.distance_transform_edt(mask); coords = feature.peak_local_max(dist, min_distance=4, labels=mask, exclude_border=False)
markers = np.zeros(dist.shape, int)
for i, (r, c) in enumerate(coords, 1): markers[r, c] = i
lbl = segmentation.watershed(-dist, markers, mask=mask)
n = len([p for p in measure.regionprops(lbl) if p.area >= 8]); print("nuclei detected:", n)
ov = segmentation.mark_boundaries(exposure.rescale_intensity(img/255.0), lbl, color=(0.1, 0.95, 0.1))
plt.figure(figsize=(9, 4)); plt.imshow(ov); plt.axis("off"); plt.title(f"Segmented nuclei (n={n})"); plt.tight_layout()

---
*Disclaimer: this notebook produces research and educational analysis of public data, not medical advice. Confidence and validation scores summarize evidence strength, not the probability of benefit for any individual. Every clinical decision must be made by a qualified health care provider, ideally within a clinical trial.*

*Developed by Dig Vijay Kumar Yarlagadda, [digvijayky.com](https://digvijayky.com).*